In [11]:
import os
import pandas as pd
import tensorflow as tf
from tqdm import tqdm
import base64

# 1. Setup paths correctly
input_path = "../Datasets/minecraft_data/data"
output_path = "../Datasets/minecraft_tfrecords" # Keep these separate!
os.makedirs(output_path, exist_ok=True)

# 2. Iterate safely
for root, _, files in os.walk(input_path):
    for file in files:
        if file.endswith(".parquet"):
            # The path to the REAL data
            parquet_path = os.path.join(root, file)
            
            # The path to the NEW tfrecord (different name/folder)
            tfrecord_path = os.path.join(output_path, file.replace(".parquet", ".tfrecord"))
            
            print(f"Processing: {parquet_path}")
            
            # Now we read the parquet BEFORE we open the TFRecordWriter
            try:
                df = pd.read_parquet(parquet_path, columns=["image", "text"])
            except Exception as e:
                print(f"Error reading {file}: {e}. (Is the file 0 bytes?)")
                continue
            
            # Now we write to the NEW path
            with tf.io.TFRecordWriter(tfrecord_path) as writer:
                for _, row in tqdm(df.iterrows(), total=len(df)):
                    img_bytes = base64.b64decode(row['image'])

                    feature_dict = {
                        'image': tf.train.Feature(bytes_list=tf.train.BytesList(value=[img_bytes])),
                        'caption': tf.train.Feature(bytes_list=tf.train.BytesList(value=[row['text'].encode('utf-8')]))
                    }

                    example = tf.train.Example(features=tf.train.Features(feature=feature_dict))
                    writer.write(example.SerializeToString())

Processing: ../Datasets/minecraft_data/data/train-00005-of-00007.parquet


100%|██████████| 122016/122016 [00:12<00:00, 10115.08it/s]


Processing: ../Datasets/minecraft_data/data/train-00006-of-00007.parquet


100%|██████████| 122016/122016 [00:12<00:00, 10129.31it/s]


Processing: ../Datasets/minecraft_data/data/train-00002-of-00007.parquet


...

100%|██████████| 122017/122017 [00:12<00:00, 10065.59it/s]


Processing: ../Datasets/minecraft_data/data/train-00001-of-00007.parquet


100%|██████████| 122017/122017 [00:12<00:00, 9727.78it/s] 


Processing: ../Datasets/minecraft_data/data/train-00000-of-00007.parquet


100%|██████████| 122017/122017 [00:12<00:00, 9912.04it/s] 
